In [1]:

# ============================================================
# RABTECH ACADEMY - TASK 03: DATA CLEANING
# Complete code in one Google Colab cell
# ============================================================

import pandas as pd
import numpy as np
import os
from IPython.display import display

# 1. LOAD THE PROVIDED DATASET
file_path = "/content/retail-orders-raw.csv"

if not os.path.exists(file_path):
    from google.colab import files
    print("Please upload retail-orders-raw.csv")
    uploaded = files.upload()
    file_path = next(iter(uploaded.keys()))

df = pd.read_csv(file_path)

print("=" * 60)
print("TASK 03: DATA CLEANING")
print("=" * 60)
print("\nRAW DATA SHAPE:", df.shape)
print("\nRAW COLUMN NAMES:")
print(df.columns.tolist())
print("\nFIRST FIVE ROWS:")
display(df.head())

# 2. STANDARDIZE COLUMN NAMES
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# 3. REMOVE COMPLETELY EMPTY ROWS AND COLUMNS
df.dropna(how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)

# 4. REMOVE EXTRA SPACES FROM TEXT COLUMNS
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# 5. STANDARDIZE TEXT CASE
text_columns = [
    "customer_segment", "city", "category", "payment_status"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].str.title()

# 6. CONVERT DATA TYPES
if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )

numeric_columns = [
    "quantity", "unit_price", "discount_pct"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 7. CHECK AND REMOVE DUPLICATE ROWS
duplicates_before = int(df.duplicated().sum())
df.drop_duplicates(inplace=True)

# Remove duplicate order IDs, keeping the first occurrence.
# This assumes each order_id should represent one order.
duplicate_ids_removed = 0
if "order_id" in df.columns:
    duplicate_ids_removed = int(
        df["order_id"].notna().sum()
        - df.loc[df["order_id"].notna(), "order_id"].nunique()
    )
    df.drop_duplicates(subset=["order_id"], keep="first", inplace=True)

# 8. HANDLE INVALID VALUES
if "quantity" in df.columns:
    df.loc[df["quantity"] <= 0, "quantity"] = np.nan

if "unit_price" in df.columns:
    df.loc[df["unit_price"] < 0, "unit_price"] = np.nan

if "discount_pct" in df.columns:
    # Assumes discount_pct is stored as a percentage from 0 to 100.
    df.loc[
        (df["discount_pct"] < 0) |
        (df["discount_pct"] > 100),
        "discount_pct"
    ] = np.nan

# 9. HANDLE MISSING VALUES
# Do not invent missing dates, cities, or customer details.
# Fill numeric missing values only where a median can be calculated.
for col in ["quantity", "unit_price", "discount_pct"]:
    if col in df.columns and df[col].notna().any():
        df[col] = df[col].fillna(df[col].median())

# Missing discount can reasonably mean no discount only if that
# is consistent with the dataset's business rules. Here, leave it
# missing rather than assume it equals zero.

# 10. CREATE SALES FEATURES
if {"quantity", "unit_price"}.issubset(df.columns):
    df["gross_sales"] = df["quantity"] * df["unit_price"]

    if "discount_pct" in df.columns:
        df["discount_amount"] = (
            df["gross_sales"] * df["discount_pct"] / 100
        )
        df["net_sales"] = df["gross_sales"] - df["discount_amount"]
    else:
        df["discount_amount"] = 0.0
        df["net_sales"] = df["gross_sales"]

if "order_date" in df.columns:
    df["year"] = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.month
    df["month_name"] = df["order_date"].dt.month_name()

# 11. DATA QUALITY CHECKS
missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percentage": [
        round(df[c].isna().mean() * 100, 2) for c in df.columns
    ]
})

quality_summary = pd.DataFrame({
    "Metric": [
        "Rows in original dataset",
        "Rows after cleaning",
        "Columns after cleaning",
        "Exact duplicate rows found before removal",
        "Duplicate order IDs removed",
        "Missing cells remaining",
        "Exact duplicate rows remaining"
    ],
    "Value": [
        # Read the original file again to report its actual dimensions.
        pd.read_csv(file_path).shape[0],
        len(df),
        len(df.columns),
        duplicates_before,
        duplicate_ids_removed,
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
})

# 12. SAVE OUTPUT FILES
clean_path = "/content/clean_dataset.csv"
missing_path = "/content/task3_missing_values_report.csv"
quality_path = "/content/task3_data_quality_summary.csv"

df.to_csv(clean_path, index=False)
missing_report.to_csv(missing_path, index=False)
quality_summary.to_csv(quality_path, index=False)

# 13. DISPLAY RESULTS
print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
display(quality_summary)

print("\nMISSING VALUES REPORT:")
display(missing_report)

print("\nCLEANED DATASET:")
display(df)

print("\nOUTPUT FILES CREATED:")
print(clean_path)
print(missing_path)
print(quality_path)

# 14. DOWNLOAD OUTPUT FILES
from google.colab import files

files.download(clean_path)
files.download(missing_path)
files.download(quality_path)

print("\nTASK 03 CLEANING CODE COMPLETED.")


TASK 03: DATA CLEANING

RAW DATA SHAPE: (12, 9)

RAW COLUMN NAMES:
['order_id', 'order_date', 'customer_segment', 'city', 'category', 'quantity', 'unit_price', 'discount_pct', 'payment_status']

FIRST FIVE ROWS:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid



CLEANING SUMMARY


,Metric,Value
0,Rows in original dataset,12
1,Rows after cleaning,11
2,Columns after cleaning,15
3,Exact duplicate rows found before removal,1
4,Duplicate order IDs removed,0
5,Missing cells remaining,13
6,Exact duplicate rows remaining,0



MISSING VALUES REPORT:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


TASK 03 CLEANING CODE COMPLETED.
